In [1]:
'''
Feature Engineering: 特征工程
date: 2026-02-25
target: 构建和选择特征，为模型训练提供高质量输入
'''


'\nFeature Engineering: 特征工程\ndate: 2026-02-25\ntarget: 构建和选择特征，为模型训练提供高质量输入\n'

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

In [3]:
# 读取原始数据
df = pd.read_csv("../data/raw/online_data.csv")
print(f"原始数据形状: {df.shape}")
print(f"原始列名: {df.columns.tolist()}\n")

# 
print(df.head())

# 保存原始数据的备份
original_count = len(df)

原始数据形状: (11429826, 7)
原始列名: ['User_id', 'Merchant_id', 'Action', 'Coupon_id', 'Discount_rate', 'Date_received', 'Date']

    User_id  Merchant_id  Action  Coupon_id Discount_rate  Date_received  \
0  13740231        18907       2  100017492        500:50     20160513.0   
1  13740231        34805       1        NaN           NaN            NaN   
2  14336199        18907       0        NaN           NaN            NaN   
3  14336199        18907       0        NaN           NaN            NaN   
4  14336199        18907       0        NaN           NaN            NaN   

         Date  
0         NaN  
1  20160321.0  
2  20160618.0  
3  20160618.0  
4  20160618.0  


### 列名标准化 & 数据类型转换

In [4]:
# 列名标准化
# 统一小写下划线命名
df.columns = df.columns.str.lower()
print(f"标准化后列名: {df.columns.tolist()}")

标准化后列名: ['user_id', 'merchant_id', 'action', 'coupon_id', 'discount_rate', 'date_received', 'date']


In [5]:
# 数据类型转换
# 转换日期字段
# 原始数据格式为 20160513.0 (float)，需要先转为字符串并去除小数部分
def first_date_conversion(series):
    # 1. 提取不重复的日期值
    uniaque_dates = series.dropna().unique()
    # 2. 仅对唯一值进行转换：先转 int 去除 .0，再转 str，最后转 datetime
    # 如果是 float 且包含 NaN，直接 astype(int) 会报错，所以用 map
    date_map = {d: pd.to_datetime(str(int(d)), format='%Y%m%d', errors='coerce') for d in uniaque_dates}
    # 3. 将转换好的日期映射会Series
    return series.map(date_map)
df['date_received'] = first_date_conversion(df['date_received'])
df['date'] = first_date_conversion(df['date'])
# 提取月份
df['month'] = df['date'].dt.month
# 提取星期几
df['weekday'] = df['date'].dt.dayofweek

# date_received 和 date 字段的范围
if df['date'].notna().any():
    print(f"  Date范围:")
    print(f"    {df['date'].min()} ~ {df['date'].max()}")
if df['date_received'].notna().any():
    print(f"  Date_received范围:")
    print(f"    {df['date_received'].min()} ~ {df['date_received'].max()}")

  Date范围:
    2016-01-01 00:00:00 ~ 2016-06-30 00:00:00
  Date_received范围:
    2016-01-01 00:00:00 ~ 2016-06-15 00:00:00


In [6]:
# 转换ID字段为字符串
df['user_id'] = df['user_id'].astype(str)
df['merchant_id'] = df['merchant_id'].astype(str)

# Coupon_id和discount_rate可能有缺失，特殊处理
df['coupon_id'] = df['coupon_id'].fillna('').astype(str)
df['discount_rate'] = df['discount_rate'].fillna('').astype(str)

# 将空字符串替换为None（数据库的NULL）
df['coupon_id'] = df['coupon_id'].replace('', None)
df['discount_rate'] = df['discount_rate'].replace('', None)

print("\n处理后数据预览:")
print(df.head())


处理后数据预览:
    user_id merchant_id  action  coupon_id discount_rate date_received  \
0  13740231       18907       2  100017492        500:50    2016-05-13   
1  13740231       34805       1       None          None           NaT   
2  14336199       18907       0       None          None           NaT   
3  14336199       18907       0       None          None           NaT   
4  14336199       18907       0       None          None           NaT   

        date  month  weekday  
0        NaT    NaN      NaN  
1 2016-03-21    3.0      0.0  
2 2016-06-18    6.0      5.0  
3 2016-06-18    6.0      5.0  
4 2016-06-18    6.0      5.0  


In [7]:
# 特征计算配置
FEATURE_END_DATE = pd.Timestamp('2016-03-31')  # 特征窗口：1-3月
TARGET_MONTHS = [4, 5, 6]  # 预测目标
TIME_WINDOWS = [7, 14, 30]  # 时间窗口

print("配置:")
print(f"  特征窗口: 2016-01-01 ~ {FEATURE_END_DATE}")
print(f"  预测目标: {TARGET_MONTHS}月")
print(f"  时间窗口: {TIME_WINDOWS}天")

配置:
  特征窗口: 2016-01-01 ~ 2016-03-31 00:00:00
  预测目标: [4, 5, 6]月
  时间窗口: [7, 14, 30]天


In [8]:
# 训练数据：1-3月
train_data = df[df['date'] <= FEATURE_END_DATE].copy()
print("训练数据(1-3月):")
print(f"记录数: {len(train_data):,} 条")
print(f"用户数: {train_data['user_id'].nunique():,}")
print(f"商户数: {train_data['merchant_id'].nunique():,}")

训练数据(1-3月):
记录数: 4,825,901 条
用户数: 544,307
商户数: 7,817


In [9]:
# 创建标记行为
# 0: 点击，1: 购买（date不为空），2: 领券（date_received不为空），3: 用券购买（date和coupon_id都不为空）
train_data['is_click'] = (train_data['action'] == 0).astype(int) if 'action' in train_data.columns else 0
# 购买：date不为空表示购买行为
train_data['is_purchase'] = train_data['date'].notna().astype(int)
# 领券: date_received不为空表示领券行为
train_data['is_coupon'] = train_data['date_received'].notna().astype(int)
# 用券购买: date和coupon_id都不为空表示用券购买行为
train_data['is_coupon_purchase'] = (
    train_data['date'].notna() & train_data['coupon_id'].notna()
).astype(int)
# 无券购买: date不为空且coupon_id为空表示无券购买行为
train_data['is_no_coupon_purchase'] = (
    train_data['date'].notna() & train_data['coupon_id'].isna()
).astype(int)
# 领券未使用: date为空且coupon_id不为空表示领券未使用行为
train_data['is_coupon_not_used'] = (
    train_data['date_received'].notna() & train_data['date'].isna()
).astype(int)

print(f"  点击: {train_data['is_click'].sum():,}")
print(f"  购买: {train_data['is_purchase'].sum():,}")
print(f"  领券: {train_data['is_coupon'].sum():,}")
print(f"  用券购买: {train_data['is_coupon_purchase'].sum():,}")
print(f"  无券购买: {train_data['is_no_coupon_purchase'].sum():,}")
print(f"  领券未使用: {train_data['is_coupon_not_used'].sum():,}")

  点击: 4,230,896
  购买: 4,825,901
  领券: 103,340
  用券购买: 103,340
  无券购买: 4,722,561
  领券未使用: 0


### 特征构建 

#### 用户基本特征
- total_purchases: 总购买次数
- total_coupons: 总领券次数
- coupon_purchases: 用券购买次数
- no_coupon_purchases: 无券购买次数
- coupons_not_used: 领券未使用次数
- merchant_count: 访问商户数
- coupon_types_count: 领券种类数
- total_actions: 总行为数

In [10]:
# 获取所有唯一用户ID，作为特征表的基础
user_features = pd.DataFrame({'user_id': train_data['user_id'].unique()})

In [11]:
# 按用户分组统计各项指标
user_basic = train_data.groupby('user_id').agg({
    'is_purchase': 'sum',
    'is_coupon': 'sum',
    'is_coupon_purchase': 'sum',
    'is_no_coupon_purchase': 'sum',
    'is_coupon_not_used': 'sum',
    'merchant_id': 'nunique',
    'coupon_id': 'nunique'
}).rename(columns={
    'is_purchase': 'total_purchases',
    'is_coupon': 'total_coupons',
    'is_coupon_purchase': 'coupon_purchases',
    'is_no_coupon_purchase': 'no_coupon_purchases',
    'is_coupon_not_used': 'coupons_not_used',
    'merchant_id': 'merchant_count',
    'coupon_id': 'coupon_types_count'
})

# 用户总行为数（所有记录条数，包括点击、购买、领券）
# 业务含义：总活跃度，行为越多说明用户越活跃
user_basic['total_actions'] = train_data.groupby('user_id').size()
# 合并到特征表
user_features = user_features.merge(user_basic, on='user_id', how='left').fillna(0)

print(f"基础统计特征: {len(user_basic.columns)} 个")

基础统计特征: 8 个


#### 用户转换率特征
- coupon_use_rate: 券核销率
- coupon_abandon_rate: 券弃用率
- coupon_purchase_ratio: 用券购买占比
- no_coupon_purchase_ratio: 无券购买占比
- purchases_per_merchant: 人均商户购买数
- coupon_diversity: 券多样性
- action_intensity: 行为活跃度
- purchase_rate: 购买转化率

In [12]:
# 1. 券核销率：用券购买 / 总领券
user_features['coupon_use_rate'] = (
    user_features['coupon_purchases'] / 
    user_features['total_coupons'].replace(0, 1)  # 避免除以0
)

# 2. 券弃用率：领券未使用 / 总领券
user_features['coupon_abandon_rate'] = (
    user_features['coupons_not_used'] / 
    user_features['total_coupons'].replace(0, 1)
)

# 3. 用券购买占比：用券购买 / 总购买
user_features['coupon_purchase_ratio'] = (
    user_features['coupon_purchases'] / 
    user_features['total_purchases'].replace(0, 1)
)

# 4. 无券购买占比：无券购买 / 总购买
user_features['no_coupon_purchase_ratio'] = (
    user_features['no_coupon_purchases'] / 
    user_features['total_purchases'].replace(0, 1)
)

# 5. 人均商户购买数：总购买 / 商户数
user_features['purchases_per_merchant'] = (
    user_features['total_purchases'] / 
    user_features['merchant_count'].replace(0, 1)
)

# 6. 券多样性：领券种类数 / 总领券数
user_features['coupon_diversity'] = (
    user_features['coupon_types_count'] / 
    user_features['total_coupons'].replace(0, 1)
)

# 7. 行为活跃度：总行为数 / 商户数
user_features['action_intensity'] = (
    user_features['total_actions'] / 
    user_features['merchant_count'].replace(0, 1)
)

# 8. 购买转化率：总购买 / 总行为数
user_features['purchase_rate'] = (
    user_features['total_purchases'] / 
    user_features['total_actions'].replace(0, 1)
)

#### RFM特征
- recency: 最近购买距今天数（R）
- frequency: 购买频率（F）
- lifetime: 用户生命周期天数
- purchase_frequency_daily: 日均购买频率
- is_new_user: 是否新用户

In [13]:
'''
计算RFM特征
用于通过用户的历史购买时间线来建模用户的活跃度和黏性
1、筛选购买行为并聚合
2、计算关键时间特征
3、用户分层示例:
- R↓ F↑ M↑ = 重要价值客户（刚买过，买得多）
- R↑ F↑ M↑ = 重要挽留客户（很久没买，但历史买得多）
- R↓ F↓ M↓ = 新用户（刚注册，买得少）
'''
# 只统计有购买记录的用户的时间特征
purchase_time = train_data[train_data['date'].notna()].groupby('user_id')['date'].agg([
    'min',   # 首次购买日期
    'max',   # 最近购买日期
    'count'  # 购买次数
])
purchase_time.columns = ['first_purchase_date', 'last_purchase_date', 'frequency']

# R: Recency - 最近一次购买距今天数
purchase_time['recency'] = (FEATURE_END_DATE - purchase_time['last_purchase_date']).dt.days

# L: Lifetime - 用户生命周期（首次到最后购买的天数）
purchase_time['lifetime'] = (
    purchase_time['last_purchase_date'] - purchase_time['first_purchase_date']
).dt.days + 1  # +1避免当天注册当天购买时lifetime=0

# 日均购买频率：购买次数 / 生命周期
purchase_time['purchase_frequency_daily'] = (
    purchase_time['frequency'] / purchase_time['lifetime']
)

# 是否新用户：生命周期 <= 7天
purchase_time['is_new_user'] = (purchase_time['lifetime'] <= 7).astype(int)

# 合并到特征表
user_features = user_features.merge(
    purchase_time[['recency', 'frequency', 'lifetime', 
                   'purchase_frequency_daily', 'is_new_user']], 
    on='user_id', 
    how='left'
)

# 填充缺失值：没有购买记录的用户
# recency=999: 表示"从未购买"
# 其他=0: 没有购买历史
user_features.fillna({
    'recency': 999, 
    'frequency': 0, 
    'lifetime': 0, 
    'purchase_frequency_daily': 0, 
    'is_new_user': 0
}, inplace=True)

#### 时间窗口特征
-

In [14]:
for window in TIME_WINDOWS:
    # 计算窗口起始日期
    window_start = FEATURE_END_DATE - timedelta(days=window)
    
    # 筛选窗口内的数据
    window_data = train_data[
        (train_data['date'] >= window_start) & 
        (train_data['date'] <= FEATURE_END_DATE)
    ]
    
    # 统计窗口内的各项指标
    window_stats = window_data.groupby('user_id').agg({
        'is_purchase': 'sum',           # 购买次数
        'is_coupon': 'sum',              # 领券次数
        'is_coupon_purchase': 'sum',     # 用券购买次数
        'is_no_coupon_purchase': 'sum',  # 无券购买次数
        'merchant_id': 'nunique',        # 访问商户数
        'date': 'count'                  # 总行为数
    }).rename(columns={
        'is_purchase': f'purchases_{window}d',
        'is_coupon': f'coupons_{window}d',
        'is_coupon_purchase': f'coupon_purchases_{window}d',
        'is_no_coupon_purchase': f'no_coupon_purchases_{window}d',
        'merchant_id': f'merchants_{window}d',
        'date': f'actions_{window}d'
    })
    
    # 合并到特征表
    user_features = user_features.merge(window_stats, on='user_id', how='left').fillna(0)
    
    # 计算窗口内的转化率
    # 窗口内券核销率
    user_features[f'coupon_use_rate_{window}d'] = (
        user_features[f'coupon_purchases_{window}d'] / 
        user_features[f'coupons_{window}d'].replace(0, 1)
    )
    
    # 窗口内购买转化率
    user_features[f'purchase_rate_{window}d'] = (
        user_features[f'purchases_{window}d'] / 
        user_features[f'actions_{window}d'].replace(0, 1)
    )

print(f"\n时间窗口特征: {len(TIME_WINDOWS) * 8} 个")


时间窗口特征: 24 个


#### 优惠券特征
- avg_coupon_use_days: 平均领券到使用天数
- median_coupon_use_days: 中位数
- std_coupon_use_days: 标准差
- min_coupon_use_days: 最快使用天数
- max_coupon_use_days: 最慢使用天数
- has_coupon_habit: 是否有领券习惯
- coupon_activity: 领券活跃度
- recent_coupon: 最近是否领券

In [15]:
# 筛选出用券购买的记录（既有领券日期又有消费日期）
coupon_used = train_data[
    train_data['date'].notna() & 
    train_data['date_received'].notna()
].copy()

if len(coupon_used) > 0:
    # 计算领券到使用的天数
    coupon_used['days_to_use'] = (
        coupon_used['date'] - coupon_used['date_received']
    ).dt.days
    
    # 统计每个用户领券到使用时间的各种指标
    coupon_time = coupon_used.groupby('user_id')['days_to_use'].agg([
        'mean',    # 平均天数
        'median',  # 中位数（更稳健，不受极端值影响）
        'std',     # 标准差（衡量稳定性）
        'min',     # 最快使用（最冲动的一次）
        'max'      # 最慢使用（最犹豫的一次）
    ]).rename(columns={
        'mean': 'avg_coupon_use_days',
        'median': 'median_coupon_use_days',
        'std': 'std_coupon_use_days',
        'min': 'min_coupon_use_days',
        'max': 'max_coupon_use_days'
    })
    
    # 合并到特征表
    user_features = user_features.merge(coupon_time, on='user_id', how='left')
    # 填充缺失值：没有用券记录的用户填999（表示从未使用）
    user_features[coupon_time.columns] = user_features[coupon_time.columns].fillna(999)

# 是否有领券习惯（0/1特征）
user_features['has_coupon_habit'] = (user_features['total_coupons'] > 0).astype(int)

# 领券活跃度：领券次数 / 总行为数
user_features['coupon_activity'] = (
    user_features['total_coupons'] / 
    user_features['total_actions'].replace(0, 1)
)

# 最近7天是否领券（0/1特征）
recent_coupon = train_data[
    (train_data['date_received'] >= FEATURE_END_DATE - timedelta(days=7)) &
    (train_data['date_received'] <= FEATURE_END_DATE)
]
recent_coupon_users = recent_coupon['user_id'].unique()
user_features['recent_coupon'] = user_features['user_id'].isin(recent_coupon_users).astype(int)

#### 折扣率特征
- avg_discount: 平均折扣力度
- median_discount: 中位数
- std_discount: 标准差
- min_discount: 最小折扣
- max_discount: 最大折扣
- discount_coupon_count: 使用折扣券次数
- discount_sensitivity: 折扣敏感度

In [16]:
def parse_discount(rate):
    """
    解析discount_rate字段
    返回:折扣力度
    """
    if pd.isna(rate):
        return np.nan
    
    rate = str(rate)
    
    if ':' in rate:
        # 满减类型：如"100:10"表示满100减10
        # 计算折扣力度：10/100 = 0.1（减免10%）
        parts = rate.split(':')
        try:
            discount_value = float(parts[1]) / float(parts[0])
            return discount_value
        except:
            return np.nan
    else:
        # 折扣类型：如"0.9"表示9折
        # 转换为折扣力度：1-0.9 = 0.1（打了9折，便宜了10%）
        try:
            discount_rate = float(rate)
            return 1 - discount_rate
        except:
            return np.nan

# 筛选有折扣率信息的记录
user_discounts = train_data[train_data['discount_rate'].notna()].copy()

if len(user_discounts) > 0:
    # 解析折扣率
    user_discounts['discount_value'] = user_discounts['discount_rate'].apply(parse_discount)
    
    # 统计每个用户使用的折扣情况
    discount_stats = user_discounts.groupby('user_id')['discount_value'].agg([
        'mean',    # 平均折扣力度
        'median',  # 中位折扣力度
        'std',     # 折扣力度标准差（稳定性）
        'min',     # 最小折扣力度（最不划算的一次）
        'max',     # 最大折扣力度（最划算的一次）
        'count'    # 使用折扣券次数
    ]).rename(columns={
        'mean': 'avg_discount',
        'median': 'median_discount',
        'std': 'std_discount',
        'min': 'min_discount',
        'max': 'max_discount',
        'count': 'discount_coupon_count'
    })
    
    # 合并到特征表
    user_features = user_features.merge(discount_stats, on='user_id', how='left')
    
    # 填充缺失值：没有使用折扣券的用户填0
    user_features[discount_stats.columns] = user_features[discount_stats.columns].fillna(0)

# 折扣敏感度：使用折扣券次数 / 总领券数
# 业务含义：
# 接近1 = 领的券都有折扣信息 = 专门找折扣
# 接近0 = 很多券没折扣信息 = 不太在意折扣
user_features['discount_sensitivity'] = (
    user_features['discount_coupon_count'] / 
    user_features['total_coupons'].replace(0, 1)
)

#### 时间偏好特征
- weekday_purchases: 工作日购买次数
- weekend_purchases: 周末购买次数
- weekday_purchase_ratio: 工作日购买占比
- weekend_purchase_ratio: 周末购买占比
- purchased_last_week: 最近7天是否购买
- purchased_last_month: 最近30天是否购买

In [17]:
weekday_purchases = train_data[
    (train_data['date'].notna()) & 
    (train_data['weekday'] < 5)
].groupby('user_id').size()

# 统计周末购买次数（周六周日，weekday >= 5）
weekend_purchases = train_data[
    (train_data['date'].notna()) & 
    (train_data['weekday'] >= 5)
].groupby('user_id').size()

# 合并到特征表
user_features = user_features.merge(
    weekday_purchases.rename('weekday_purchases'), 
    on='user_id', 
    how='left'
).fillna(0)

user_features = user_features.merge(
    weekend_purchases.rename('weekend_purchases'), 
    on='user_id', 
    how='left'
).fillna(0)

# 工作日购买占比
user_features['weekday_purchase_ratio'] = (
    user_features['weekday_purchases'] / 
    user_features['total_purchases'].replace(0, 1)
)

# 周末购买占比
user_features['weekend_purchase_ratio'] = (
    user_features['weekend_purchases'] / 
    user_features['total_purchases'].replace(0, 1)
)

# 最近7天是否购买（0/1特征）
recent_purchase = train_data[
    (train_data['date'] >= FEATURE_END_DATE - timedelta(days=7)) &
    (train_data['date'] <= FEATURE_END_DATE) &
    (train_data['date'].notna())
]
recent_buyers = recent_purchase['user_id'].unique()
user_features['purchased_last_week'] = user_features['user_id'].isin(recent_buyers).astype(int)

# 最近30天是否购买（0/1特征）
recent_month_purchase = train_data[
    (train_data['date'] >= FEATURE_END_DATE - timedelta(days=30)) &
    (train_data['date'] <= FEATURE_END_DATE) &
    (train_data['date'].notna())
]
recent_month_buyers = recent_month_purchase['user_id'].unique()
user_features['purchased_last_month'] = user_features['user_id'].isin(recent_month_buyers).astype(int)

#### 商户交互特征
- main_merchant_id: 主要商户ID
- main_merchant_purchases: 主商户购买次数
- main_merchant_purchase_ratio: 主商户购买占比
- merchant_loyalty: 商户忠诚度
- merchant_exploration: 商户探索度
- avg_purchases_per_merchant: 人均商户购买次数

In [18]:
# 查找每个用户最常访问的商户
# mode()[0]：取出现次数最多的商户ID
user_main_merchant = train_data.groupby('user_id')['merchant_id'].agg(
    lambda x: x.mode()[0] if len(x.mode()) > 0 else None
).reset_index()
user_main_merchant.columns = ['user_id', 'main_merchant_id']

# 统计用户在主要商户的购买次数
# 先合并主要商户ID
main_merchant_purchases = train_data.merge(user_main_merchant, on='user_id')

# 筛选：在主要商户的购买记录
main_merchant_purchases = main_merchant_purchases[
    (main_merchant_purchases['merchant_id'] == main_merchant_purchases['main_merchant_id']) &
    (main_merchant_purchases['date'].notna())
].groupby('user_id').size().rename('main_merchant_purchases')

# 合并到特征表
user_features = user_features.merge(user_main_merchant, on='user_id', how='left')
user_features = user_features.merge(main_merchant_purchases, on='user_id', how='left').fillna(0)

# 主要商户购买占比：主商户购买数 / 总购买数
user_features['main_merchant_purchase_ratio'] = (
    user_features['main_merchant_purchases'] / 
    user_features['total_purchases'].replace(0, 1)
)

# 商户忠诚度：主商户购买数 / 商户总数
user_features['merchant_loyalty'] = (
    user_features['main_merchant_purchases'] / 
    user_features['merchant_count'].replace(0, 1)
)

# 商户探索度：商户数 / log(购买数+1)
user_features['merchant_exploration'] = (
    user_features['merchant_count'] / 
    np.log1p(user_features['total_purchases'])  # log1p = log(x+1)，避免log(0)
)

# 平均每商户购买次数
user_features['avg_purchases_per_merchant'] = (
    user_features['total_purchases'] / 
    user_features['merchant_count'].replace(0, 1)
)

## 创建训练数据集

In [19]:
# 排除user_id和main_merchant_id，统计特征列
feature_cols = [col for col in user_features.columns 
                if col not in ['user_id', 'main_merchant_id']]


In [20]:
# 为每个目标月份创建训练数据集
for target_month in TARGET_MONTHS:
    print(f"\n{target_month}月:")
    
    # 获取目标月份的购买用户（正样本）
    # 只要在目标月份有过购买，就标记为1
    target_users = df[
        (df['month'] == target_month) & 
        (df['date'].notna())
    ]['user_id'].unique()
    
    # 复制特征表，添加标签
    data = user_features.copy()
    data['label'] = data['user_id'].isin(target_users).astype(int)
    
    # 统计正负样本
    pos = data['label'].sum()
    neg = (data['label'] == 0).sum()
    pos_rate = pos / len(data) * 100
    
    print(f"  正样本（{target_month}月购买）: {pos:,} ({pos_rate:.2f}%)")
    print(f"  负样本（{target_month}月未购买）: {neg:,}")
    print(f"  正负样本比: 1:{neg/pos:.1f}")
    
    # 保存为CSV文件
    output = f'../data/features/user_features_month{target_month}.csv'
    data[['user_id'] + feature_cols + ['label']].to_csv(output, index=False)


4月:
  正样本（4月购买）: 261,280 (48.00%)
  负样本（4月未购买）: 283,027
  正负样本比: 1:1.1

5月:
  正样本（5月购买）: 257,987 (47.40%)
  负样本（5月未购买）: 286,320
  正负样本比: 1:1.1

6月:
  正样本（6月购买）: 257,397 (47.29%)
  负样本（6月未购买）: 286,910
  正负样本比: 1:1.1


## 保存特征列表

In [21]:
feature_list_file = '../data/features/feature_list.txt'
with open(feature_list_file, 'w', encoding='utf-8') as f:
    f.write(f"# 特征列表\n")
    f.write(f"# 生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"# 总特征数: {len(feature_cols)}\n")
    f.write(f"# 预测目标: 4/5/6月用户复购\n\n")
    
    # 按类别写入特征
    categories = {
        '基础统计': feature_cols[:8],
        '转化率': feature_cols[8:16],
        'RFM': feature_cols[16:21],
        '时间窗口': feature_cols[21:21+len(TIME_WINDOWS)*8],
        '优惠券': feature_cols[21+len(TIME_WINDOWS)*8:29+len(TIME_WINDOWS)*8],
        '折扣': feature_cols[29+len(TIME_WINDOWS)*8:35+len(TIME_WINDOWS)*8],
        '时间偏好': feature_cols[35+len(TIME_WINDOWS)*8:41+len(TIME_WINDOWS)*8],
        '商户交互': feature_cols[41+len(TIME_WINDOWS)*8:]
    }
    
    idx = 1
    for category, features in categories.items():
        f.write(f"\n## {category}特征 ({len(features)}个)\n")
        for feat in features:
            f.write(f"{idx}. {feat}\n")
            idx += 1